In [1]:
import pandas as pd
import os, dill
import numpy as np
import cvxpy as cp
import plotly.express as px
import plotly.graph_objects as go

from ecoli.processes.metabolism_redux_classic import NetworkFlowModel, FlowResult

os.chdir(os.path.expanduser('~/dev/vEcoli'))

%reload_ext autoreload


In [2]:
def load_sim(
        folder_path:str,
):
    """ This function is meant to load an output of a simulation in timeseries form.
        Note: This is not designed for parquet output format.

    """
    # --- Load Sim ---
    output = np.load(folder_path + '0_output.npy',allow_pickle='TRUE').item()
    output = output['agents']['0']

    fba = output['listeners']['fba_results']
    bulk = pd.DataFrame(output['bulk'])
    f = open(folder_path + 'agent_steps.pkl', 'rb')
    agent = dill.load(f)
    f.close()

    metabolism = agent['ecoli-metabolism-redux-classic']

    return fba, bulk, metabolism, output

# Define helper functions for standalone FBA

In [3]:
def get_subset_S(S, met_of_interest):
    S_met = S.loc[met_of_interest, :]
    S_met = S_met.loc[:,~np.all(S_met == 0, axis=0)]
    return S_met, S_met.columns

def get_keys(dict, value):
    return [key for key in dict if np.any(np.isin(value, dict[key]))]

def plot_unmet_need(estimated_dm_dt, homeostatic_metabolites, homeostatic_dm_targets, homeostatic_concs, top_n=10):
    """Bar chart of the top_n homeostatic metabolites by unmet need for a single
    FBA solve: |target_dm_dt - estimated_dm_dt| / homeostatic_concs.

    Recreates the top-subplot summary from `WC_metabolite_unmet_need` in
    20260127_WC_unmet_need_homeo_analysis.ipynb (the dmdt_diff/met_score cell),
    adapted for one standalone solve instead of a multi-timestep sim -- there's
    no time axis here, so only the top-N bar chart carries over, not the bottom
    per-metabolite timeseries subplot.
    """
    unmet_need = np.abs(np.asarray(homeostatic_dm_targets) - np.asarray(estimated_dm_dt)) / np.asarray(homeostatic_concs)
    scores = pd.Series(unmet_need, index=list(homeostatic_metabolites)).sort_values(ascending=False)
    top = scores.head(top_n)

    pastel = px.colors.qualitative.Pastel
    fig = go.Figure(
        go.Bar(
            x=top.index,
            y=top.values,
            marker=dict(color=[pastel[i % len(pastel)] for i in range(len(top))]),
            text=[f"{v:.1e}" for v in top.values],
            name="Unmet Homeostatic Need",
        )
    )
    fig.update_yaxes(title_text="Unmet Need", type="log", tickformat=".0e")
    fig.update_xaxes(title_text="Metabolite")
    fig.update_layout(
        title=f"Top {top_n} metabolites by unmet homeostatic need",
        template="plotly_white",
    )
    fig.show()
    return fig, scores

def test_NetworkFlowModel(objective_weights, metabolism, fba,
                          uptake_addition = set([]),
                          uptake_removal = set([]),
                          new_exchange_molecules = set([]),
                          add_metabolite = None,
                          add_reaction = None,
                          remove_reaction = None,
                          add_kinetic = None,
                          force_reaction = None,
                          add_homeostatic_demand = None,
                          top_n_unmet = 10,
                          plot = False,
                          solver_choice=cp.GLOP):

    # --- Update Exchanges ---
    # Note: Exchange molecules are the molecules that can be secreted
    #       Uptake molecules are the molecules that can be taken up.
    uptake = metabolism.allowed_exchange_uptake.copy()
    uptake = set(uptake)
    uptake = uptake | uptake_addition
    uptake = uptake - uptake_removal

    exchange_molecules = metabolism.exchange_molecules.copy()
    exchange_molecules = exchange_molecules | new_exchange_molecules

    # --- Get whole-cell model outputs for FBA ---
    reaction_names = metabolism.reaction_names.copy()
    metabolites = metabolism.metabolite_names.copy()

    kinetic_reaction_ids = metabolism.kinetic_constraint_reactions.copy()

    kinetic = pd.DataFrame(fba["target_kinetic_fluxes"],
                           columns=metabolism.kinetic_constraint_reactions).copy().loc[500,:] # units in counts/s
    homeostatic = pd.DataFrame(fba["target_homeostatic_dmdt"],
                               columns=metabolism.homeostatic_metabolites).loc[500,:] # units in counts/s
    homeostatic_counts = pd.DataFrame(fba["homeostatic_metabolite_counts"],
                                      columns=metabolism.homeostatic_metabolites).loc[500, :] # units in counts
    maintenance = pd.DataFrame(fba["maintenance_target"][1:], columns=['maintenance_reaction']).iat[500, 0]

    S_new = metabolism.stoichiometry.copy()

    # --- Add/remove reactions, metabolites, kinetic constraints, homeostatic demands ---
    if add_metabolite is not None: #
        for m in add_metabolite:
            if m not in metabolites:
                metabolites.append(m)
        # append rows of zeros to S_new of length add_metabolite
        S_new = np.concatenate((S_new, np.zeros((len(add_metabolite), S_new.shape[1]))), axis=0)

    if add_reaction is not None:
        # Note: if one wants to add a reaction, it should be in a dictionary of
        #       form {"reaction_name": {"metabolite1": stoich1, "metabolite2": stoich2, ...}}
        assert isinstance(add_reaction, dict)

        for r,s in add_reaction.items():
            assert isinstance(s, dict)
            if r not in reaction_names:
                reaction_names.append(r)
            # append columns of reaction stoich to S_new of length add_reaction
            new_reaction = np.zeros((S_new.shape[0], 1))
            for m, v in s.items():
                new_reaction[metabolites.index(m), 0] = v
            S_new = np.concatenate((S_new, new_reaction), axis=1)

    if add_kinetic is not None:
        # Note: if one wants to add kinetics, it should be in a dictionary of
        #       form {"reaction_name": kinetic_target}
        assert isinstance(add_kinetic, dict)

        for r, v in add_kinetic.items():
            if r not in kinetic_reaction_ids:
                kinetic_reaction_ids.append(r)
                kinetic[r] = v
            if r in kinetic_reaction_ids:
                kinetic[r] = v

    if remove_reaction is not None:
        for r in remove_reaction:
            r_idx = reaction_names.index(r)
            S_new = np.delete(S_new, r_idx, axis=1)
            reaction_names.remove(r)
            if r in kinetic_reaction_ids:
                kinetic_reaction_ids.remove(r)
                del kinetic[r]

    if force_reaction is not None:
        force_reaction_idx = np.array([reaction_names.index(r) for r in force_reaction])
    else:
        force_reaction_idx = force_reaction

    if add_homeostatic_demand is not None:
        # assert add_homeostatic_demand is a list
        assert isinstance(add_homeostatic_demand, list)

        for met in add_homeostatic_demand:
            # here I just arbitrarily set the homeostatic demand to be 100 counts/s,
            # and homeostatic metabolite count to be 1, but this can be changed as needed.
            homeostatic[met] = 100
            homeostatic_counts[met] = 1

    # --- Convert every count-based FBA input to concentration ---
    # Everything above (homeostatic/homeostatic_counts/kinetic/maintenance) is in
    # raw molecule counts (or counts/s), straight from the fba listener. The live
    # whole-cell sim solves NetworkFlowModel in concentration units with
    # upper_flux_bound=100 (the class default); previously this notebook instead
    # left everything in raw counts and hacked upper_flux_bound up to 10^9 to
    # compensate, which (per debugging) both inflates absolute magnitudes
    # unpredictably and lets a few "hub" metabolites with huge raw dm/dt swamp the
    # homeostatic objective regardless of nutrient condition. Multiplying every
    # count-based quantity by the same counts_to_molar factor keeps the LP fully
    # concentration-scale and consistent, matching the live sim's own units.
    counts_to_molar = metabolism.counts_to_molar.asNumber()
    homeostatic_concs_conc = homeostatic_counts.values * counts_to_molar
    homeostatic_dm_targets_conc = np.array(list(dict(homeostatic).values())) * counts_to_molar
    kinetic_targets_conc = np.array(list(dict(kinetic).values())) * counts_to_molar
    maintenance_conc = maintenance * counts_to_molar

    # --- Solve NetworkFlowModel ---
    model = NetworkFlowModel(
            stoich_arr=S_new,
            metabolites=metabolites,
            reactions=reaction_names,
            homeostatic_metabolites=metabolism.homeostatic_metabolites,
            kinetic_reactions=kinetic_reaction_ids)
    model.set_up_exchanges(exchanges=exchange_molecules, uptakes=uptake)
    solution: FlowResult = model.solve(
            homeostatic_concs=homeostatic_concs_conc,
            homeostatic_dm_targets=homeostatic_dm_targets_conc,
            maintenance_target=maintenance_conc,
            kinetic_targets=kinetic_targets_conc,
            binary_kinetic_idx=metabolism.binary_kinetic_idx,
            # binary_kinetic_idx=None,
            force_flow_idx=force_reaction_idx,
            objective_weights=objective_weights,
            upper_flux_bound=100, # matches the live WC sim's scale now everything is in concentration units.
            solver=solver_choice) #SCS. ECOS, MOSEK

    # --- Plot unmet homeostatic need for this run (see plot_unmet_need above) ---
    if plot:
        estimated_dm_dt = np.asarray(solution.dm_dt)[model.homeostatic_idx]
        plot_unmet_need(
            estimated_dm_dt, metabolism.homeostatic_metabolites,
            homeostatic_dm_targets_conc, homeostatic_concs_conc, top_n=top_n_unmet,
        )

    return solution.objective, solution.velocities, reaction_names, S_new, metabolites, kinetic, solution, model


# Load Sim and run standalone FBA

In [4]:
time_num = 100
date = '2026-07-07'
experiment_name = 'basal_original_weights'
condition = 'basal'
experiment_type = 'phenotypic'

time = str(time_num)
entry = f'{experiment_name}_{time}_{date}'
folder_path = f'out/{experiment_type}/{entry}/'
# ^^^ The above is just where I stored my sim output.

folder_path = f'out/{experiment_type}/basal_no_new_reactions_original_weights_2500_2026-07-09/'
fba, bulk, metabolism, output = load_sim(folder_path)

In [74]:
# objective_weights = {'secretion': 0.01, 'efficiency': 1e-06, 'kinetics': 1e-05, 'diversity': 1e-07, 'homeostatic': 1}
objective_weights = {'secretion': 1e-06, 'efficiency': 1e-06, 'kinetics': 1e-07, 'diversity': 1e-07, 'homeostatic': 1.0}

# Simulate how flux would look like if we were to add a new reaction
# Note: This can be used if we want to simulate certain gene additions
# add_reaction={
#     'TEMP-REACTION-1': {
#         'APO-CITRATE-LYASE[c]' : 1,
#     },
#     'TEMP-REACTION-2': {
#         'CIT[c]':-1,
#         'CITRATE-LYASE[c]' : -1,
#     }
# }

add_reaction = {}
# Our model has a problem of not being able to partition flow.
# I often use the force_reaction argument to see if the model would still
# solve if we were to force some reaction to sustain flux.
# I use this argument to disentangle dead-end metabolites and reactions leading to them.
# force_reaction = ['OXALODECARB-RXN']

# Simulate how the flux would look like if using citrate instead of glucose as the carbon source
uptake_addition = set(['L-ARABINOSE[p]'])
uptake_removal = set(['GLC[p]', 'CA+2[p]'])

# Simulate how flux would look like if we remove a downstream reaction of citrate degradation
# Note: This can be used if we want to simulate certain gene knockouts --
#       by removing the reactions that the gene/enzyme is involved in.
# remove_reaction = ['CITTRANS-RXN']

In [75]:
result = test_NetworkFlowModel(objective_weights,
                               metabolism=metabolism,
                               fba=fba,
                               # uptake_addition=uptake_addition,
                               # uptake_removal=uptake_removal,
                               # force_reaction=force_reaction,
                               # add_reaction=add_reaction,
                               # remove_reaction=remove_reaction,
                               plot = True
)

[oofv, solution_flux, test_reaction_names, S_new, metabolites_new, kinetics_new, solution, model] = result


In [63]:
solution.homeostatic_term, oofv

(np.float64(0.48521593280184705), np.float64(0.49933480718269535))

In [65]:
solution.homeostatic_term, oofv

(np.float64(0.5235187229255033), np.float64(0.5339475674969679))

In [67]:
solution.homeostatic_term, oofv

(np.float64(0.485443503986974), np.float64(0.4999310229334589))

In [69]:
solution.homeostatic_term, oofv

(np.float64(0.5235187229255033), np.float64(0.5339475674969679))

In [272]:
solution.homeostatic_term, oofv

(np.float64(14.27977664968397), np.float64(479.8142437076346))

In [256]:
sim_flux = pd.DataFrame({
    'flux': solution_flux,
    'is_new': [
        'TEMP' if id in add_reaction.keys()
        else 'Original Reactions'
            for id in test_reaction_names
    ]
}, index=test_reaction_names)

This concludes running the stand-alone FBA. Now we can do various analysis on this output

In [257]:
# For example, we can look at the fluxes of reactions involving a certain metabolite of interest.
met_of_interest = ['GLC[p]', 'L-ARABINOSE[p]', 'BETA-L-ARABINOSE[c]']
S_new = pd.DataFrame(S_new, index=metabolites_new, columns=test_reaction_names)

S_met, rxns  = get_subset_S(S_new, met_of_interest)

kinetic_reaction_ids = metabolism.kinetic_constraint_reactions.copy()
sim_flux['kinetic'] = [kinetics_new[r] if r in kinetic_reaction_ids else None for r in sim_flux.index]
rxn_flux = sim_flux.loc[rxns]
rxn_flux

,flux,is_new,kinetic
3.2.1.21-RXN-Beta-D-glucosides/WATER//Non-Glucosylated-Glucose-Acceptors/GLC.64.,-0.0,Original Reactions,NaN
ABC-2-RXN,0.0,Original Reactions,NaN
ABC-2-RXN-ATP/ARABINOSE/WATER//ADP/BETA-L-ARABINOSE/Pi/PROTON.52.,-0.0,Original Reactions,NaN
ABC-2-RXN-ATP/BETA-L-ARABINOSE/WATER//ADP/BETA-L-ARABINOSE/Pi/PROTON.59.,-0.0,Original Reactions,NaN
ABC-2-RXN-ATP/CPD-12045/WATER//ADP/BETA-L-ARABINOSE/Pi/PROTON.52.,-0.0,Original Reactions,NaN
...,...,...,...
TRANS-RXN-40-CPD-12045/PROTON//L-ARABINOSE/PROTON.37.,-0.0,Original Reactions,NaN
TRANS-RXN-40-CPD-12046/PROTON//L-ARABINOSE/PROTON.37.,0.0,Original Reactions,NaN
TRANS-RXN-40-CPD-15699/PROTON//L-ARABINOSE/PROTON.37.,-0.0,Original Reactions,NaN
TRANS-RXN-40-L-ARABINOSE/PROTON//L-ARABINOSE/PROTON.39.,0.0,Original Reactions,NaN


In [265]:
solution.homeostatic_term, oofv

(np.float64(14.27977664968397), np.float64(479.8142437076346))

In [259]:
solution.homeostatic_term, oofv

(np.float64(14.27977664968397), np.float64(479.8142437076346))

In [260]:
solution.homeostatic_term, oofv

(np.float64(14.27977664968397), np.float64(479.8142437076346))

# Check homeostatic metabolite

In [261]:
homeostatic_dmdt = solution.dm_dt[metabolism.network_flow_model.homeostatic_idx]
homeostatic_metabolites = np.array(metabolism.homeostatic_metabolites)
homeostatic_counts = pd.DataFrame(fba["homeostatic_metabolite_counts"],
                                     columns=metabolism.homeostatic_metabolites).mean(axis=0).copy()
homeostatic_df = pd.DataFrame({
    'metabolite': homeostatic_metabolites,
    'dmdt': homeostatic_dmdt,
    'target_dmdt': np.array(fba["target_homeostatic_dmdt"][1:]).mean(axis=0),
    'counts': homeostatic_counts
})
homeostatic_df

,metabolite,dmdt,target_dmdt,counts
2-3-DIHYDROXYBENZOATE[c],2-3-DIHYDROXYBENZOATE[c],34.9296,34.9296,122675.3592
2-KETOGLUTARATE[c],2-KETOGLUTARATE[c],89.3484,89.3484,313800.0204
2-PG[c],2-PG[c],23.2360,23.2360,81605.7836
2K-4CH3-PENTANOATE[c],2K-4CH3-PENTANOATE[c],34.9296,34.9296,122675.3592
4-AMINO-BUTYRATE[c],4-AMINO-BUTYRATE[c],76.9464,76.9464,270241.3892
...,...,...,...,...
NA+[p],NA+[p],25.3112,25.3112,88895.2080
OXYGEN-MOLECULE[p],OXYGEN-MOLECULE[p],25.3364,25.3364,88895.1828
FE+3[p],FE+3[p],25.3228,25.3228,88895.1964
CA+2[p],CA+2[p],25.3412,25.3412,88895.1780


In [262]:
homeostatic_df

,metabolite,dmdt,target_dmdt,counts
2-3-DIHYDROXYBENZOATE[c],2-3-DIHYDROXYBENZOATE[c],34.9296,34.9296,122675.3592
2-KETOGLUTARATE[c],2-KETOGLUTARATE[c],89.3484,89.3484,313800.0204
2-PG[c],2-PG[c],23.2360,23.2360,81605.7836
2K-4CH3-PENTANOATE[c],2K-4CH3-PENTANOATE[c],34.9296,34.9296,122675.3592
4-AMINO-BUTYRATE[c],4-AMINO-BUTYRATE[c],76.9464,76.9464,270241.3892
...,...,...,...,...
NA+[p],NA+[p],25.3112,25.3112,88895.2080
OXYGEN-MOLECULE[p],OXYGEN-MOLECULE[p],25.3364,25.3364,88895.1828
FE+3[p],FE+3[p],25.3228,25.3228,88895.1964
CA+2[p],CA+2[p],25.3412,25.3412,88895.1780


In [263]:
sum(abs(homeostatic_df['dmdt']-homeostatic_df['target_dmdt'])/homeostatic_df['counts'])

14.279776649683969

In [264]:
sum(abs(homeostatic_df['dmdt']-homeostatic_df['target_dmdt'])/homeostatic_df['counts'])

14.279776649683969

#### Bar plot of fluxes of reactions involving the metabolite of interest

In [253]:
# Top 10 reactions by absolute flux
top10_flux = (
    sim_flux.assign(abs_flux=sim_flux["flux"].abs())
    .sort_values("abs_flux", ascending=False)
    .head(10)
    .copy()
)

# Keep labels in plotting order (largest first)
top10_flux = top10_flux.sort_values("abs_flux", ascending=True)

fig = px.bar(
    top10_flux,
    x="abs_flux",
    y=top10_flux.index,
    orientation="h",
    color="flux",  # preserves sign information
    color_continuous_scale="RdBu",
    labels={"abs_flux": "|Flux|", "y": "Reaction", "flux": "Signed flux"},
    title="Top 10 Reactions by Flux Magnitude",
)

fig.update_layout(
    yaxis_title="Reaction",
    xaxis_title="Absolute flux",
    template="plotly_white",
    height=500,
)
fig.update_xaxes(type="log")
fig.show()

### Scatterplot of the reactions with kinetic target

In [254]:
# Keep only reactions with a defined kinetic target
plot_flux = sim_flux.copy()
# plot_flux = rxn_flux.copy()

kinetic_compare = (
    plot_flux[["flux", "kinetic"]]
    .dropna(subset=["kinetic"])
    .rename(columns={"flux": "simulated_flux", "kinetic": "target_kinetic_flux"})
    .copy()
)

kinetic_compare['simulated_flux'] += 1e-9
kinetic_compare['target_kinetic_flux'] += 1e-9

# If there are no kinetic targets in this subset, this will print and skip plotting
if kinetic_compare.empty:
    print("No reactions in rxn_flux have kinetic targets.")
else:
    # Build symmetric range for y=x reference line
    min_val = min(
        kinetic_compare["simulated_flux"].min(),
        kinetic_compare["target_kinetic_flux"].min(),
    )
    max_val = max(
        kinetic_compare["simulated_flux"].max(),
        kinetic_compare["target_kinetic_flux"].max(),
    )

    fig = px.scatter(
        kinetic_compare.reset_index().rename(columns={"index": "reaction"}),
        x="target_kinetic_flux",
        y="simulated_flux",
        hover_data=["reaction"],
        title="Simulated vs Target Kinetic Flux (Reactions with Kinetic Targets)",
        labels={
            "target_kinetic_flux": "Target kinetic flux (count/s)",
            "simulated_flux": "Simulated flux (count/s)",
        },
        template="plotly_white",
    )

    # Add y = x line (perfect agreement)
    fig.add_shape(
        type="line",
        x0=min_val,
        y0=min_val,
        x1=max_val,
        y1=max_val,
        line=dict(color="gray", dash="dash"),
    )

    fig.update_layout(height=500)
    fig.update_xaxes(type="log")
    fig.update_yaxes(type="log")
    fig.show()

# test old script

In [50]:
def get_subset_S(S, met_of_interest):
    S_met = S.loc[met_of_interest, :]
    S_met = S_met.loc[:,~np.all(S_met == 0, axis=0)]
    return S_met, S_met.columns

def get_keys(dict, value):
    return [key for key in dict if dict[key] == value]

def test_NetworkFlowModel(objective_weights,
                          uptake_addition = set([]), uptake_removal = set([]), new_exchange_molecules = set([]),
                          add_metabolite = None, add_reaction = None, add_kinetic = None, remove_reaction = None, force_reaction = None, solver_choice=cp.GLOP):
    # update exchanges
    uptake = metabolism.allowed_exchange_uptake.copy()
    uptake = set(uptake)
    uptake = uptake | uptake_addition
    uptake = uptake - uptake_removal

    exchange_molecules = metabolism.exchange_molecules.copy()
    exchange_molecules = exchange_molecules | new_exchange_molecules

    # update stoichiometry
    reaction_names = metabolism.reaction_names.copy()
    kinetic_reaction_ids = metabolism.kinetic_constraint_reactions.copy()
    kinetic = pd.DataFrame(fba["target_kinetic_fluxes"], columns=metabolism.kinetic_constraint_reactions).loc[24, :].copy()
    metabolites = metabolism.metabolite_names.copy()

    S_new = stoichiometry.copy()

    if add_metabolite is not None: # add to metabolites list because they are currently not included in the model
        for m in add_metabolite:
            if m not in metabolites:
                metabolites.append(m)
        # append rows of zeros to S_new of length add_metabolite
        S_new = np.concatenate((S_new, np.zeros((len(add_metabolite), S_new.shape[1]))), axis=0)

    if add_reaction is not None:
        # assert add_reaction is a dictionary
        assert isinstance(add_reaction, dict)

        for r,s in add_reaction.items():
            if r not in reaction_names:
                reaction_names.append(r)
            # append columns of reaction stoich to S_new of length add_reaction
            new_reaction = np.zeros((S_new.shape[0], 1))
            for m, v in s.items():
                new_reaction[metabolites.index(m), 0] = v
            S_new = np.concatenate((S_new, new_reaction), axis=1)

    if add_kinetic is not None:
        # assert add_kinetic is a dictionary
        assert isinstance(add_kinetic, dict)

        for r, v in add_kinetic.items():
            if r not in kinetic_reaction_ids:
                kinetic_reaction_ids.append(r)
                kinetic[r] = v

    if remove_reaction is not None:
        for r in remove_reaction:
            r_idx = reaction_names.index(r)
            S_new = np.delete(S_new, r_idx, axis=1)
            reaction_names.remove(r)
            if r in kinetic_reaction_ids:
                kinetic_reaction_ids.remove(r)
                del kinetic[r]

    if force_reaction is not None:
        force_reaction_idx = np.array([reaction_names.index(r) for r in force_reaction])
    else:
        force_reaction_idx = force_reaction

    # Solve NetworkFlowModel
    model = NetworkFlowModel(
            stoich_arr=S_new,
            metabolites=metabolites,
            reactions=reaction_names,
            homeostatic_metabolites=metabolism.homeostatic_metabolites,
            kinetic_reactions=kinetic_reaction_ids,
            free_reactions=FREE_RXNS)
    model.set_up_exchanges(exchanges=exchange_molecules, uptakes=uptake)
    solution: FlowResult = model.solve(
            homeostatic_concs=homeostatic_count * metabolism.counts_to_molar.asNumber(), # in conc
            homeostatic_dm_targets=np.array(list(dict(homeostatic).values())), # *10^7
            maintenance_target=maintenance, # *10^6 ish
            kinetic_targets=np.array(list(dict(kinetic).values())), # *10^6 ish
            # binary_kinetic_idx=binary_kinetic_idx, #7646
            binary_kinetic_idx=None,
            force_flow_idx=force_reaction_idx,
            objective_weights=objective_weights, #same
            upper_flux_bound= 1000000000, # increase to 10^9 because notebook runs FlowResult using Counts, WC runs using conc.
            solver=solver_choice) #SCS. ECOS, MOSEK
    return solution.objective, solution.velocities, reaction_names, S_new, metabolites, kinetic, solution

In [6]:
folder = f'out/phenotypic/basal_original_weights_1000_2026-07-07/'

output = np.load(folder + '0_output.npy',allow_pickle='TRUE').item()
# output = np.load(r"out/geneRxnVerifData/output_glc.npy", allow_pickle=True, encoding='ASCII').tolist()
output = output['agents']['0']
fba = output['listeners']['fba_results']
bulk = pd.DataFrame(output['bulk'])
f = open(folder + 'agent_steps.pkl', 'rb')
agent = dill.load(f)
f.close()

In [8]:
# get commonly stored variables
metabolism = agent['ecoli-metabolism-redux-classic']
stoichiometry = metabolism.stoichiometry.copy()
reaction_names = metabolism.reaction_names
fba_new_reaction_ids = metabolism.parameters["fba_new_reaction_ids"]
fba_reaction_ids_to_base_reaction_ids = metabolism.parameters['fba_reaction_ids_to_base_reaction_ids']
metabolites = metabolism.metabolite_names.copy()
binary_kinetic_idx = metabolism.binary_kinetic_idx
exchange_molecules = metabolism.exchange_molecules

S = stoichiometry .copy()
S = pd.DataFrame(S, index=metabolites , columns=reaction_names )
homeostatic_count = pd.DataFrame(fba["homeostatic_metabolite_counts"], columns=metabolism.homeostatic_metabolites).loc[24, :]
homeostatic = pd.DataFrame(fba["target_homeostatic_dmdt"], columns=metabolism.homeostatic_metabolites).loc[24, :]
maintenance = pd.DataFrame(fba["maintenance_target"][1:], columns=['maintenance_reaction']).iat[24, 0]
kinetic = pd.DataFrame(fba["target_kinetic_fluxes"], columns=metabolism.kinetic_constraint_reactions).loc[24, :].copy()

In [9]:
# microarray plate 1: ~ tests 96 carbon sources
conditions = {
    'A1 - Carbon Negative Control - MIX0-80': {
        'Add': set([]),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'A2 - L-Arabinose* - MIX0-420': {
        'Add': set(['L-ARABINOSE[p]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'A3 - N-Acetyl-D- Glucosamine* - MIX0-421': {
        'Add': set(['N-acetyl-D-glucosamine[p]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'A4 - D-Saccharic acid - MIX0-422': {
        'Add': set(['D-GLUCARATE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'A5 - Succinic acid - MIX0-423': {
        'Add': set(['SUC[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'A6 - D-Galactose* - MIX0-424': {
        'Add': set(['GALACTOSE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'A7 - L-Aspartic acid - MIX0-425 ': {
        'Add': set(['L-ASPARTATE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'A8 - L-Proline - MIX0-426': {
        'Add': set(['PRO[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'A9 - D-Alanine - MIX0-427': {
        'Add': set(['D-ALANINE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'A10 - D-Trehalose - MIX0-428': {
        'Add': set(['TREHALOSE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'A11 - D-Mannose - MIX0-429': {
        'Add': set(['CPD-13559[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'A12 - Dulcitol - MIX0-430': {
        'Add': set(['GALACTITOL[p]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'B1 - D-Serine - MIX0-431': {
        'Add': set(['D-SERINE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'B2 - D-Sorbitol - MIX0-432': {
        'Add': set(['SORBITOL[e]']),
        'Remove': set(['GLC[p]','CA+2[p]']),
    },
    'B3 - Glycerol - MIX0-433': {
        'Add': set(['GLYCEROL[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'B4 - L-Fucose* - MIX0-434': {
        'Add': set(['L-fucoses[p]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'B5 - D-Glucuronic acid* - MIX0-435': {
        'Add': set(['CPD-15530[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'B6 - D-Gluconic acid - MIX0-436': {
        'Add': set(['GLUCONATE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'B7 - DL-α- Glycerol Phosphate - MIX0-437': {
        'Add': set(['GLYCEROL-3P[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'B8 - D-Xylose* - MIX0-438': {
        'Add': set(['CPD-15377[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'B9 - L-Lactic acid - MIX0-439': {
        'Add': set(['L-LACTATE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'B10 - Formic acid - MIX0-440': {
        'Add': set(['FORMATE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'B11 - D-Mannitol - MIX0-441': {
        'Add': set(['MANNITOL[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'B12 - L-Glutamic acid - MIX0-442': {
        'Add': set(['GLT[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'C1 - D-Glucose- 6-Phosphate* - MIX0-443': {
        'Add': set(['GLC-6-P[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'C2 - D-Galactonic acid-γ- Lactone - MIX0-444': {
        'Add': set(['D-GALACTONO-1-4-LACTONE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'C3 - DL-Malic acid - MIX0-445': {
        'Add': set(['MAL[e]', 'CPD-660[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'C4 - D-Ribose* - MIX0-446': {
        'Add': set(['CPD0-1110[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'C5 - Tween 20 - MIX0-797' : None, # not in the model
    'C6 - L-Rhamnose* - MIX0-447': {
        'Add': set(['RHAMNOSE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'C7 - D-Fructose - MIX0-448': {
        'Add': set(['BETA-D-FRUCTOSE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'C8 - Acetic acid - MIX0-449': {
        'Add': set(['ACET[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'C9 - α-D-Glucose - MIX0-450': {
        'Add': set(['ALPHA-GLUCOSE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'C10 - Maltose - MIX0-451': {
        'Add': set(['MALTOSE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'C11 - D-Melibiose - MIX0-452': {
        'Add': set(['MELIBIOSE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'C12 - Thymidine - MIX0-453': {
        'Add': set(['THYMIDINE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'D1 - L-Asparagine - MIX0-454': {
        'Add': set(['ASN[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'D2 - D-Aspartic acid - MIX0-455': {
        'Add': set(['CPD-302[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'D3 - D-Glucosaminic acid - MIX0-456': {
        'Add': set(['GLUCOSAMINATE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'D4 - 1,2-Propanediol - MIX0-457': {
        'Add': set(['PROPANE-1-2-DIOL[c]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'D5 - Tween 40 - MIX0-798': None, # not in the model
    'D6 - α-Ketoglutaric acid - MIX0-458': {
        'Add': set(['2-KETOGLUTARATE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'D7 - α-Ketobutyric acid - MIX0-459': {
        'Add': set(['2-OXOBUTANOATE[p]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'D8 - α-Methyl-D- Galactoside - MIX0-786': {
        'Add': set(['CPD-3565[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'D9 - α-D-Lactose - MIX0-460': {
        'Add': set(['Alpha-lactose[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'D10 - Lactulose - MIX0-461': {
        'Add': set(['CPD-3561[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'D11 - Sucrose - MIX0-462': {
        'Add': set(['SUCROSE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'D12 - Uridine - MIX0-463': {
        'Add': set(['URIDINE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'E1 - L-Glutamine - MIX0-464': {
        'Add': set(['GLN[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'E2 - M-Tartaric acid - MIX0-465': None, # not in the model
    'E3 - D-Glucose- 1-Phosphate - MIX0-466': {
        'Add': set(['GLC-1-P[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'E4 - D-Fructose- 6-Phosphate - MIX0-467': {
        'Add': set(['FRUCTOSE-6P[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'E5 - Tween 80 - MIX0-799': None, # not in the model
    'E6 - α-Hydroxy glutaric_acid γ-Lactone - MIX0-793': {
        'Add': set(['CPD-13414[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'E7 - α-Hydroxy butyric_acid - MIX0-790': {
        'Add': set(['CPD-3564[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'E8 - β-Methyl-D- Glucoside - MIX0-784': {
        'Add': set(['CPD-3570[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'E9 - Adonitol - MIX0-468': {
        'Add': set(['RIBITOL[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'E10 - Maltotriose - MIX0-469': {
        'Add': set(['MALTOTRIOSE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'E11 - 2-Deoxyadenosine - MIX0-470': {
        'Add': set(['DEOXYADENOSINE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'E12 - Adenosine - MIX0-471': {
        'Add': set(['ADENOSINE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'F1 - Gly-Asp - MIX0-778': {
        'Add': set(['CPD-13406[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'F2 - Citric acid - MIX0-472': {
        'Add': set(['CIT[p]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'F3 - M-Inositol  - MIX0-473': {
        'Add': set(['MYO-INOSITOL[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    }, # lacks uptake and metabolic pathway
    'F4 - D-Threonine - MIX0-474': {
        'Add': set(['D-THREONINE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'F5 - Fumaric acid - MIX0-475': {
        'Add': set(['FUM[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'F6 - Bromosuccinic acid - MIX0-779': None, # not in the model
    'F7 - Propionic acid - MIX0-476': {
        'Add': set(['PROPIONATE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'F8 - Mucic acid - MIX0-477': {
        'Add': set(['D-GALACTARATE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'F9 - Glycolic acid - MIX0-478': {
        'Add': set(['GLYCOLLATE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'F10 - Glyoxylic acid - MIX0-479': {
        'Add': set(['GLYOX[p]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'F11 - D-Cellobiose - MIX0-480': {
        'Add': set(['CELLOBIOSE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'F12 - Inosine - MIX0-481': {
        'Add': set(['INOSINE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'G1 - Gly-Glu - MIX0-482': {
        'Add': set(['CPD-3569[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'G2 - Tricarballylic acid - MIX0-483': None, # not in the model
    'G3 - L-Serine - MIX0-484': {
        'Add': set(['SER[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'G4 - L-Threonine - MIX0-485': {
        'Add': set(['THR[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'G5 - L-Alanine - MIX0-486': {
        'Add': set(['L-ALPHA-ALANINE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'G6 - Ala-Gly - MIX0-772': {
        'Add': set(['ALA-GLY[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'G7 - Acetoacetic acid - MIX0-487': {
        'Add': set(['3-KETOBUTYRATE[p]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'G8 - N-Acetyl-D Mannosamine* - MIX0-488': {
        'Add': set(['N-acetyl-D-mannosamine[p]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'G9 - Mono- Methylsuccinate - MIX0-489': None, # not in the model
    'G10 - Methyl pyruvate - MIX0-490': None, # not in the model
    'G11 - D-Malic acid - MIX0-491': {
        'Add': set(['CPD-660[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'G12 - L-Malic acid - MIX0-492': {
        'Add': set(['MAL[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'H1 - Gly-Pro - MIX0-493': {
        'Add': set(['CPD-10814[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'H2 - p-Hydroxy phenyl Acetic_acid - MIX0-494': {
        'Add': set(['4-HYDROXYPHENYLACETATE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    }, # lacks uptake and metabolic pathway
    'H3 - m-Hydroxy phenyl Acetic_acid - MIX0-495': {
        'Add': set(['3-HYDROXYPHENYLACETATE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    }, # lacks uptake and metabolic pathway
    'H4 - Tyramine - MIX0-496': {
        'Add': set(['TYRAMINE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    }, # lacks uptake and metabolic pathway
    'H5 - D-Psicose - MIX0-497': {
        'Add': set(['Ket0-D-Psicose[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    }, # lacks uptake and metabolic pathway
    'H6 - L-Lyxose - MIX0-498': {
        'Add': set(['L-LYXOSE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'H7 - Glucuronamide - MIX0-499': None, # not in the model
    'H8 - Pyrunic acid - MIX0-500': {
        'Add': set(['PYRUVATE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'H9 - L-Galactonic acid-γ- Lactone - MIX0-794': {
        'Add': set(['CPD-330[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'H10 - D-Galacturonic acid* - MIX0-501': {
        'Add': set(['CPD-15633[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'H11 - Phenyl ethylamine - MIX0-502': {
        'Add': set(['PHENYLETHYLAMINE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
    'H12 - 2-Aminoethanol - MIX0-503': {
        'Add': set(['ETHANOL-AMINE[e]']),
        'Remove': set(['GLC[p]', 'CA+2[p]']),
    },
}

In [22]:
# run tests
FREE_RXNS = ["TRANS-RXN-145", "TRANS-RXN0-545", "TRANS-RXN0-474"]
df_all = pd.DataFrame()
condition_names = []
cp3_oofv = dict({})
plateID_to_condition = dict({})
for condition_name, condition in conditions.items():

    # store in dictionary the optimal objective function value
    temp = condition_name.split(' - ')
    plate_ID = temp[0]
    carbon_source = temp[1]
    plateID_to_condition[plate_ID] = carbon_source

    # solve the cvxpy problem
    objective_weights = {'secretion': 0.01, 'efficiency': 1e-06, 'kinetics': 1e-05, 'diversity': 1e-07, 'homeostatic': 1}
    if condition == None:
        cp3_oofv[plate_ID] = None
        continue

    try:
        oofv, solution_flux, test_reaction_names, S_new, test_metabolites, test_kinetic = test_NetworkFlowModel(
                                            objective_weights,
                                            uptake_addition=condition['Add'], uptake_removal=condition['Remove'],
                                            solver_choice=cp.GLOP,)
    except:
        print(f"Error in solving for condition: {condition_name}")
        cp3_oofv[plate_ID] = None
        continue

    # get the fluxes
    sim_flux = pd.DataFrame({f'sim_cp3_{condition_name}': solution_flux}, index = test_reaction_names)
    condition_names.append(f'sim_cp3_{condition_name}')
    df_all = pd.concat([df_all, sim_flux], axis=1)
    cp3_oofv[plate_ID] = oofv

    print(f"""Finished enviornment: {condition_name} with objective function value: {oofv}""")

Finished enviornment: A1 - Carbon Negative Control - MIX0-80 with objective function value: 48797.402150789516
Finished enviornment: A2 - L-Arabinose* - MIX0-420 with objective function value: 23721.04741163886
Finished enviornment: A3 - N-Acetyl-D- Glucosamine* - MIX0-421 with objective function value: 21292.776111882893
Finished enviornment: A4 - D-Saccharic acid - MIX0-422 with objective function value: 33311.80422114747


/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


Finished enviornment: A5 - Succinic acid - MIX0-423 with objective function value: 35420.54383971795
Finished enviornment: A6 - D-Galactose* - MIX0-424 with objective function value: 23677.87930949084
Error in solving for condition: A7 - L-Aspartic acid - MIX0-425 
Finished enviornment: A8 - L-Proline - MIX0-426 with objective function value: 20177.486953487263
Finished enviornment: A9 - D-Alanine - MIX0-427 with objective function value: 23756.820820769757
Finished enviornment: A10 - D-Trehalose - MIX0-428 with objective function value: 23476.393237646265
Finished enviornment: A11 - D-Mannose - MIX0-429 with objective function value: 23476.291775571965
Finished enviornment: A12 - Dulcitol - MIX0-430 with objective function value: 21569.265831430464
Error in solving for condition: B1 - D-Serine - MIX0-431
Finished enviornment: B2 - D-Sorbitol - MIX0-432 with objective function value: 21179.29246175138
Finished enviornment: B3 - Glycerol - MIX0-433 with objective function value: 16231.8

/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


Finished enviornment: C2 - D-Galactonic acid-γ- Lactone - MIX0-444 with objective function value: 26252.382716631735
Finished enviornment: C3 - DL-Malic acid - MIX0-445 with objective function value: 40134.00832686097
Finished enviornment: C4 - D-Ribose* - MIX0-446 with objective function value: 24459.60064508729
Finished enviornment: C6 - L-Rhamnose* - MIX0-447 with objective function value: 23806.312211524968


...

Error in solving for condition: E7 - α-Hydroxy butyric_acid - MIX0-790
Finished enviornment: E8 - β-Methyl-D- Glucoside - MIX0-784 with objective function value: 26838.172563015378
Finished enviornment: E9 - Adonitol - MIX0-468 with objective function value: 48797.402150789516
Finished enviornment: E10 - Maltotriose - MIX0-469 with objective function value: 23219.82480693442
Finished enviornment: E11 - 2-Deoxyadenosine - MIX0-470 with objective function value: 19381.269575705148
Error in solving for condition: E12 - Adenosine - MIX0-471
Error in solving for condition: F1 - Gly-Asp - MIX0-778
Error in solving for condition: F2 - Citric acid - MIX0-472
Finished enviornment: F3 - M-Inositol  - MIX0-473 with objective function value: 48797.402150789516
Finished enviornment: F4 - D-Threonine - MIX0-474 with objective function value: 48797.402150789516
Finished enviornment: F5 - Fumaric acid - MIX0-475 with objective function value: 40136.008568676596
Error in solving for condition: F7 - Pro

/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


Finished enviornment: G1 - Gly-Glu - MIX0-482 with objective function value: 28911.918370272917
Finished enviornment: G3 - L-Serine - MIX0-484 with objective function value: 28809.652505160702
Finished enviornment: G4 - L-Threonine - MIX0-485 with objective function value: 21794.96606190898
Finished enviornment: G5 - L-Alanine - MIX0-486 with objective function value: 23755.841108012006
Finished enviornment: G6 - Ala-Gly - MIX0-772 with objective function value: 26835.345776103713
Finished enviornment: G7 - Acetoacetic acid - MIX0-487 with objective function value: 23991.58413020994
Finished enviornment: G8 - N-Acetyl-D Mannosamine* - MIX0-488 with objective function value: 21293.157738103528


/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


Error in solving for condition: G11 - D-Malic acid - MIX0-491
Error in solving for condition: G12 - L-Malic acid - MIX0-492
Finished enviornment: H1 - Gly-Pro - MIX0-493 with objective function value: 22102.135841430718
Finished enviornment: H2 - p-Hydroxy phenyl Acetic_acid - MIX0-494 with objective function value: 48797.402150789516
Finished enviornment: H3 - m-Hydroxy phenyl Acetic_acid - MIX0-495 with objective function value: 48797.402150789516
Finished enviornment: H4 - Tyramine - MIX0-496 with objective function value: 48797.40215104061
Finished enviornment: H5 - D-Psicose - MIX0-497 with objective function value: 48797.402150789516
Finished enviornment: H6 - L-Lyxose - MIX0-498 with objective function value: 23721.862101502975
Finished enviornment: H8 - Pyrunic acid - MIX0-500 with objective function value: 30390.521299468615
Finished enviornment: H9 - L-Galactonic acid-γ- Lactone - MIX0-794 with objective function value: 48797.40215092831
Finished enviornment: H10 - D-Galactur

In [14]:
carbon_source

'2-Aminoethanol'

In [21]:
objective_weights = {'secretion': 0.01, 'efficiency': 1e-06, 'kinetics': 1e-05, 'diversity': 1e-07, 'homeostatic': 1}
oofv, solution_flux, test_reaction_names, S_new, test_metabolites, test_kinetic = test_NetworkFlowModel(
                                            objective_weights,
                                            uptake_addition=condition['Add'], uptake_removal=condition['Remove'],
                                            solver_choice=cp.GLOP,)

# Some plots

In [19]:
reaction_flux = pd.DataFrame(fba['estimated_fluxes'], columns=metabolism.reaction_names).iloc[1:]
new_reaction_flux = reaction_flux[metabolism.parameters["fba_new_reaction_ids"]]
np.unique(new_reaction_flux)

array([0.])

In [21]:
import plotly.express as px
px.bar(x=[1], y=[1]).write_image("test.svg")